# Entrenamiento de Detector de Fatiga con Wav2Vec2-XLS-R

Este cuaderno realiza el **ajuste fino (fine-tuning)** del modelo multilingüe **Wav2Vec2-XLS-R-300m** para clasificar audios en dos estados:
*   **0: Alerta** (agrupa audios de happy y neutral)
*   **1: Fatiga** (agrupa audios de calm y sad)

Además, utiliza la probabilidad Softmax final para dar un **Nivel de Fatiga continuo (0% a 100%)** y clasificar el nivel de riesgo.

### Celda 1: Montar Google Drive
Conectamos Colab con tu Google Drive para acceder al dataset comprimido y guardar el modelo entrenado.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Celda 2: Instalar Dependencias
Instalamos las librerías necesarias de Hugging Face y procesamiento de audio.

In [ ]:
!pip install -q transformers datasets accelerate soundfile librosa evaluate fvcore

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00


### Celda 3: Copiar la Carpeta del Dataset en Local
Copiamos la carpeta sin comprimir desde tu Google Drive al disco rápido local de la máquina virtual de Colab. Esto acelera el entrenamiento y evita límites de lectura de Google Drive.

In [ ]:
import os

drive_dataset_path = "/content/drive/MyDrive/GABI/dataset"
local_dataset_path = "/content/dataset"

if not os.path.exists(drive_dataset_path):
    drive_dataset_path = "/content/drive/MyDrive/dataset"

if not os.path.exists(drive_dataset_path):
    print("Error: No se encontró la carpeta 'dataset' en tu Google Drive.")
    print("Por favor, verifica que esté en 'Mi Unidad/GABI/dataset' o en 'Mi Unidad/dataset'.")
else:
    print(f"Copiando dataset desde Drive: {drive_dataset_path}...")
    !cp -r {drive_dataset_path} {local_dataset_path}
    print("Dataset copiado exitosamente en /content/dataset/")

Copiando dataset desde Drive: /content/drive/MyDrive/dataset...
Dataset copiado exitosamente en /content/dataset/


### Celda 3.5: División Train/Test (80/20) y Aumentación de Datos (8x) en Local
Separamos el dataset base en **80% de entrenamiento (Train)** y **20% de validación (Test)** *antes* de realizar cualquier aumentación. Esto es un principio crítico de Machine Learning para evitar la **Fuga de Datos (Data Leakage)**. 

Posteriormente, aplicamos las transformaciones (ruido de cabina browniano, pitch shifting y speed stretching) **únicamente** sobre el conjunto de entrenamiento. El set de test se mantiene 100% original y limpio para garantizar una validación honesta.

In [ ]:
import os
import glob
import random
import shutil
import numpy as np
import librosa
import soundfile as sf
from sklearn.model_selection import train_test_split

local_dataset_path = "/content/dataset"
split_dataset_path = "/content/dataset_split"

# Limpiar directorios previos si existen
if os.path.exists(split_dataset_path):
    shutil.rmtree(split_dataset_path)

# Crear estructura de carpetas
for split in ["train", "test"]:
    for clase in ["alerta", "fatiga"]:
        os.makedirs(os.path.join(split_dataset_path, split, clase), exist_ok=True)

# 1. Obtener archivos base por clase
alerta_files = glob.glob(os.path.join(local_dataset_path, "alerta", "*.wav"))
fatiga_files = glob.glob(os.path.join(local_dataset_path, "fatiga", "*.wav"))

# 2. Hacer el split 80% train / 20% test antes de aumentar
train_alerta, test_alerta = train_test_split(alerta_files, test_size=0.2, random_state=42)
train_fatiga, test_fatiga = train_test_split(fatiga_files, test_size=0.2, random_state=42)

# Copiar archivos de test intactos (sin aumentación)
print(f"Copiando {len(test_alerta)} audios de alerta a test...")
for f in test_alerta:
    shutil.copy2(f, os.path.join(split_dataset_path, "test", "alerta"))
print(f"Copiando {len(test_fatiga)} audios de fatiga a test...")
for f in test_fatiga:
    shutil.copy2(f, os.path.join(split_dataset_path, "test", "fatiga"))

# Funciones de aumentación
def generar_ruido_cabina(length):
    white = np.random.normal(0, 1, length)
    brown = np.cumsum(white)
    brown = brown - np.mean(brown)
    brown = brown / np.max(np.abs(brown))
    return brown

def mezclar_ruido(y, noise_level=0.015):
    noise = generar_ruido_cabina(len(y))
    y_noisy = y + noise_level * noise
    return np.clip(y_noisy, -1.0, 1.0)

def aplicar_aumentacion_8x(file_list, dest_folder, label):
    print(f"Aplicando aumentación 8x a {len(file_list)} archivos de clase '{label}'...")
    for idx, filepath in enumerate(file_list):
        y, sr = librosa.load(filepath, sr=16000)
        base_name = os.path.basename(filepath).replace(".wav", "")
        
        # 1. ORIGINAL (limpio)
        sf.write(os.path.join(dest_folder, f'{base_name}_1_orig.wav'), y, sr)
        
        # 2. RUIDO DE CABINA LEVE (limpio + ruido leve)
        y_noise_low = mezclar_ruido(y, noise_level=0.01)
        sf.write(os.path.join(dest_folder, f'{base_name}_2_noise_low.wav'), y_noise_low, sr)
        
        # 3. RUIDO DE CABINA MODERADO (limpio + ruido moderado)
        y_noise_med = mezclar_ruido(y, noise_level=0.02)
        sf.write(os.path.join(dest_folder, f'{base_name}_3_noise_med.wav'), y_noise_med, sr)
        
        # 4. PITCH SHIFT AGUDO (+1.5 semitonos, limpio)
        y_pitch_up = librosa.effects.pitch_shift(y, sr=sr, n_steps=1.5)
        sf.write(os.path.join(dest_folder, f'{base_name}_4_pitch_up.wav'), y_pitch_up, sr)
        
        # 5. PITCH SHIFT GRAVE (-1.5 semitonos, limpio)
        y_pitch_down = librosa.effects.pitch_shift(y, sr=sr, n_steps=-1.5)
        sf.write(os.path.join(dest_folder, f'{base_name}_5_pitch_down.wav'), y_pitch_down, sr)
        
        # 6. TIME STRETCHING DEPENDIENTE DE LA CLASE (limpio)
        if label == "alerta":
            y_stretch = librosa.effects.time_stretch(y, rate=1.15)
        else:
            y_stretch = librosa.effects.time_stretch(y, rate=0.85)
        sf.write(os.path.join(dest_folder, f'{base_name}_6_stretch.wav'), y_stretch, sr)
        
        # 7. COMBINACIÓN A: Pitch Grave + Ruido Leve
        y_comb_a = mezclar_ruido(y_pitch_down, noise_level=0.01)
        sf.write(os.path.join(dest_folder, f'{base_name}_7_comb_grave_ruido.wav'), y_comb_a, sr)
        
        # 8. COMBINACIÓN B: Speed Stretch + Ruido Moderado
        y_comb_b = mezclar_ruido(y_stretch, noise_level=0.02)
        sf.write(os.path.join(dest_folder, f'{base_name}_8_comb_speed_ruido.wav'), y_comb_b, sr)

# Aumentar solo el conjunto de entrenamiento
aplicar_aumentacion_8x(train_alerta, os.path.join(split_dataset_path, "train", "alerta"), label="alerta")
aplicar_aumentacion_8x(train_fatiga, os.path.join(split_dataset_path, "train", "fatiga"), label="fatiga")

print(f"\n¡Aumentación completada!")
print(f"Entrenamiento final (8x): {len(train_alerta) * 8} alerta + {len(train_fatiga) * 8} fatiga = {(len(train_alerta) + len(train_fatiga)) * 8} audios.")
print(f"Test/Validación final (Original): {len(test_alerta)} alerta + {len(test_fatiga)} fatiga = {len(test_alerta) + len(test_fatiga)} audios.")

### Celda 4: Carga y Mapeo Binario del Dataset
Leemos los directorios de audio y mapeamos automáticamente todas las carpetas con la palabra `alerta` a la clase `0` (alerta) y las de `fatiga` a la clase `1` (fatiga). Luego dividimos en 80% train y 20% validation.

In [ ]:
from datasets import load_dataset, Audio

# Cargar audios automaticamente desde las carpetas de train/test locales
raw_dataset = load_dataset("audiofolder", data_dir="/content/dataset_split")

# Obtener las clases detectadas en base a las subcarpetas
detected_classes = raw_dataset["train"].features["label"].names
print("Clases detectadas por directorios:", detected_classes)

# Crear mapeo dinamico a clases binarias (0: alerta, 1: fatiga)
label_mapping = {}
for idx, name in enumerate(detected_classes):
    name_lower = name.lower()
    if "alerta" in name_lower or "happy" in name_lower or "neutral" in name_lower:
        label_mapping[idx] = 0
    elif "fatiga" in name_lower or "calm" in name_lower or "sad" in name_lower:
        label_mapping[idx] = 1

print("Mapeo de etiquetas numericas:", label_mapping)

def map_to_binary(batch):
    batch["label"] = [label_mapping[l] for l in batch["label"]]
    return batch

# Aplicar mapeo
binary_dataset = raw_dataset.map(map_to_binary, batched=True)

# El dataset ya viene pre-dividido de forma segura desde el disco en /content/dataset_split/train y test
split_dataset = binary_dataset
print("Datos de Entrenamiento:", split_dataset["train"])
print("Datos de Validacion (Test):", split_dataset["test"])

Resolving data files:   0%|          | 0/251 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Clases detectadas por directorios: ['calm', 'happy', 'neutral', 'sad']
Mapeo de etiquetas numericas: {0: 1, 1: 0, 2: 0, 3: 1}


Map:   0%|          | 0/251 [00:00<?, ? examples/s]

Datos de Entrenamiento: Dataset({
    features: ['audio', 'label'],
    num_rows: 200
})
Datos de Validacion: Dataset({
    features: ['audio', 'label'],
    num_rows: 51
})


### Celda 5: Feature Extractor de Audio (Re-muestrear a 16kHz)
Cambiamos la frecuencia de muestreo de los audios a 16kHz y extraemos las características necesarias limitando el audio a un máximo de 10 segundos.

In [ ]:
from transformers import AutoFeatureExtractor

# Cargar extractor de características del modelo XLS-R
feature_extractor = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-xls-r-300m")

# Re-muestrear audios a 16kHz
split_dataset = split_dataset.cast_column("audio", Audio(sampling_rate=16000))

def preprocess_function(examples):
    audio_inputs = [x["array"] for x in examples["audio"]]
    inputs = feature_extractor(
        audio_inputs,
        sampling_rate=feature_extractor.sampling_rate,
        # Aumentamos a 10.0 segundos para evitar recortar la fatiga (que promedia 7.66 segundos)
        max_length=int(feature_extractor.sampling_rate * 10.0),
        truncation=True
    )
    return inputs

encoded_dataset = split_dataset.map(preprocess_function, remove_columns=["audio"], batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/51 [00:00<?, ? examples/s]

### Celda 6: Inicializar Data Collator y Modelo Wav2Vec2-XLS-R
Cargamos el modelo configurando 2 clases de salida y congelamos el extractor convolucional para evitar sobreajuste.

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import AutoModelForAudioClassification

@dataclass
class DataCollatorWithPadding:
    feature_extractor: Any
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        labels = [feature["label"] for feature in features]

        batch = self.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

data_collator = DataCollatorWithPadding(feature_extractor=feature_extractor)

# Asignar etiquetas
id2label = {0: "alerta", 1: "fatiga"}
label2id = {"alerta": 0, "fatiga": 1}

model = AutoModelForAudioClassification.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

# Congelar el feature encoder
model.freeze_feature_encoder()

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
project_q.bias               | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
classifier.bias              | MISSING    | 
projector.weight             | MISSING    | 
classifier.weight            | MISSING    | 
projector.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Celda 7: Configurar Métricas de Evaluación
Definimos funciones para medir la exactitud (Accuracy) y el F1-Score durante la evaluación.

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    acc_val = accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    f1_val = f1.compute(predictions=predictions, references=labels, average="binary")["f1"]

    return {"accuracy": acc_val, "f1": f1_val}

### Celda 8: Configurar Entrenamiento y Entrenar
Configuramos los hiperparámetros óptimos y corremos el proceso de fine-tuning.

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="/content/wav2vec2-fatiga-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True, # Usar precision mixta con la GPU
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.385358,0.614516,0.843137,0.914894
2,1.253516,0.531829,0.843137,0.914894
3,1.238821,0.506223,0.843137,0.914894
4,1.274106,0.493346,0.843137,0.914894
5,1.253348,0.491843,0.843137,0.914894
6,1.156546,0.490847,0.843137,0.914894


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=78, training_loss=1.2392179048978365, metrics={'train_runtime': 819.9475, 'train_samples_per_second': 1.464, 'train_steps_per_second': 0.095, 'total_flos': 1.4470676383816426e+17, 'train_loss': 1.2392179048978365, 'epoch': 6.0})

### Celda 9: Guardar el Modelo Entrenado
Guardamos los pesos y el feature extractor final en la carpeta designada de tu Drive.

In [ ]:
# Guardar modelo y procesador localmente primero
local_save_path = "/content/modelo_fatiga_wav2vec2"
model.save_pretrained(local_save_path)
feature_extractor.save_pretrained(local_save_path)

# Copiar la carpeta final a Google Drive de forma segura
drive_save_path = "/content/drive/MyDrive/GABI/"
!cp -r {local_save_path} {drive_save_path}

print(f"¡Modelo guardado exitosamente en Drive en: {drive_save_path}modelo_fatiga_wav2vec2!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

¡Modelo guardado exitosamente en Drive en: /content/drive/MyDrive/GABI/modelo_fatiga_wav2vec2!


### Celda 10: Inferencia y Medición del Nivel de Fatiga en tiempo real
Función para probar audios arbitrarios, convirtiendo las logits a probabilidades con Softmax para obtener un porcentaje (0% a 100%) del nivel de fatiga.

In [ ]:
import os
import csv
import torch
import librosa
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor

# 1. Cargar el modelo entrenado desde Drive
saved_model_path = '/content/drive/MyDrive/GABI/modelo_fatiga_wav2vec2'
model_infer = AutoModelForAudioClassification.from_pretrained(saved_model_path)
feature_extractor_infer = AutoFeatureExtractor.from_pretrained(saved_model_path)

# 2. Cargar YAMNet desde TensorFlow Hub para detectar bostezos/suspiros
print('Cargando YAMNet desde TensorFlow Hub...')
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')

# Obtener mapeo de clases de YAMNet (contiene 521 clases, ej. Yawn, Sigh)
class_map_path = yamnet_model.class_map_path().numpy().decode('utf-8')
class_names = []
with open(class_map_path) as f:
    reader = csv.reader(f)
    next(reader)  # Saltar cabecera
    for row in reader:
        class_names.append(row[2])

def predecir_nivel_fatiga(ruta_archivo_audio):
    # --- PASO 1: CARGAR AUDIO ---
    speech, sr = librosa.load(ruta_archivo_audio, sr=16000)

    # --- PASO 2: DETECCIÓN DE EVENTOS Fisiológicos (YAMNet) ---
    scores, embeddings, spectrogram = yamnet_model(speech)
    scores = scores.numpy()

    # Obtener índices de clases clave en AudioSet (YAMNet no tiene 'Yawn' en su ontología, pero detecta bostezos como Sigh o Gasp)
    sigh_idx = class_names.index('Sigh')
    gasp_idx = class_names.index('Gasp')
    snoring_idx = class_names.index('Snoring')

    # Obtener el puntaje máximo detectado en cualquier segmento de tiempo
    max_scores = np.max(scores, axis=0)
    sigh_score = max_scores[sigh_idx]
    gasp_score = max_scores[gasp_idx]
    snoring_score = max_scores[snoring_idx]

    # Umbral de confianza de detección (0.25 es un valor de confianza alto en YAMNet)
    UMBRAL_DETECCION = 0.25

    if sigh_score > UMBRAL_DETECCION:
        nivel_fatiga = 92.0
        riesgo = 'CRÍTICO (Suspiro / Bostezo detectado)'
        color = '\033[91m'  # Rojo
        print(f'Análisis de Audio: {os.path.basename(ruta_archivo_audio)}')
        print(f'-> [YAMNet] SUSPIRO / BOSTEZO DETECTADO (Confianza: {sigh_score*100:.1f}%)')
        print(f'{color}Nivel de Fatiga: {nivel_fatiga:.2f}% -> Riesgo: {riesgo}\033[0m\n')
        return

    if gasp_score > UMBRAL_DETECCION:
        nivel_fatiga = 90.0
        riesgo = 'CRÍTICO (Jadeo / Bostezo detectado)'
        color = '\033[91m'  # Rojo
        print(f'Análisis de Audio: {os.path.basename(ruta_archivo_audio)}')
        print(f'-> [YAMNet] JADEO DETECTADO (Confianza: {gasp_score*100:.1f}%)')
        print(f'{color}Nivel de Fatiga: {nivel_fatiga:.2f}% -> Riesgo: {riesgo}\033[0m\n')
        return

    if snoring_score > UMBRAL_DETECCION:
        nivel_fatiga = 95.0
        riesgo = 'CRÍTICO (Ronquido detectado)'
        color = '\033[91m'  # Rojo
        print(f'Análisis de Audio: {os.path.basename(ruta_archivo_audio)}')
        print(f'-> [YAMNet] RONQUIDO DETECTADO (Confianza: {snoring_score*100:.1f}%)')
        print(f'{color}Nivel de Fatiga: {nivel_fatiga:.2f}% -> Riesgo: {riesgo}\033[0m\n')
        return

    # --- PASO 3: ANÁLISIS ACÚSTICO DE VOZ (Wav2Vec2) ---
    inputs = feature_extractor_infer(speech, sampling_rate=16000, return_tensors='pt')

    with torch.no_grad():
        logits = model_infer(**inputs).logits

    probabilidades = torch.softmax(logits, dim=-1).squeeze().tolist()

    prob_alerta = probabilidades[0]
    prob_fatiga = probabilidades[1]

    nivel_fatiga = prob_fatiga * 100

    # Determinar riesgo en base al porcentaje
    if nivel_fatiga < 30:
        riesgo = 'BAJO (Estado de Alerta Activo)'
        color = '\033[92m'  # Verde
    elif nivel_fatiga < 60:
        riesgo = 'LEVE (Relajación / Monitoreo)'
        color = '\033[93m'  # Amarillo
    elif nivel_fatiga < 85:
        riesgo = 'MODERADO (Se sugiere descanso corto)'
        color = '\033[94m'  # Azul
    else:
        riesgo = 'CRÍTICO (Somnolencia / Peligro Extremo)'
        color = '\033[91m'  # Rojo

    print(f'Análisis de Audio: {os.path.basename(ruta_archivo_audio)}')
    print(f'Probabilidad de Alerta: {prob_alerta*100:.2f}%')
    print(f'Probabilidad de Fatiga: {prob_fatiga*100:.2f}%')
    print(f'{color}Nivel de Fatiga: {nivel_fatiga:.2f}% -> Riesgo: {riesgo}\033[0m\n')

predecir_nivel_fatiga('/content/ElevenLabs_2026-05-31T20_31_49_Cristina Campos - Natural Conversations_pvc_sp100_s40_sb45_se0_b_e2.mp3')

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Cargando YAMNet desde TensorFlow Hub...
Análisis de Audio: ElevenLabs_2026-05-31T20_31_49_Cristina Campos - Natural Conversations_pvc_sp100_s40_sb45_se0_b_e2.mp3
-> [YAMNet] SUSPIRO / BOSTEZO DETECTADO (Confianza: 39.0%)
Nivel de Fatiga: 92.00% -> Riesgo: CRÍTICO (Suspiro / Bostezo detectado)

